In [1]:
# ============================================================
# ORZYN AI m2.0
# Notebook 06
# GitHub Issues Intelligence
# ============================================================

In [2]:
from pathlib import Path
import sys

BACKEND_DIR = Path.cwd().parent

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

In [3]:
from __future__ import annotations
from collections import Counter
from dataclasses import dataclass
from statistics import mean

import pandas as pd

from orzyn import (
    client,
    parse_datetime,
    get_active_repository,
)

In [4]:
repo_config = get_active_repository()

OWNER = repo_config.owner

REPOSITORY = repo_config.repository

print(f"Repository : {OWNER}/{REPOSITORY}")

Repository : facebook/react


In [5]:
ISSUES_QUERY = """
query(
    $owner:String!,
    $name:String!
){

repository(

    owner:$owner,

    name:$name

){

issues(

    first:100,

    orderBy:{
        field:CREATED_AT,
        direction:DESC
    }

){

nodes{

number

title

state

createdAt

closedAt

comments{

totalCount

}

author{

login

}

labels(

    first:20

){

nodes{

name

}

}

assignees(

    first:20

){

nodes{

login

}

}

}

}

}

}
"""

In [6]:
@dataclass(slots=True)
class IssueProfile:

    number: int

    title: str

    author: str

    state: str

    created_at: object

    closed_at: object | None

    comments: int

    labels: list[str]

    assignees: list[str]

In [7]:
def build_issue(node: dict) -> IssueProfile:

    author = node.get("author")

    return IssueProfile(

        number=node["number"],

        title=node["title"],

        author=author["login"] if author else "Unknown",

        state=node["state"],

        created_at=parse_datetime(
            node["createdAt"]
        ),

        closed_at=(

            parse_datetime(
                node["closedAt"]
            )

            if node["closedAt"]

            else None

        ),

        comments=node["comments"]["totalCount"],

        labels=[

            label["name"]

            for label in

            node["labels"]["nodes"]

        ],

        assignees=[

            person["login"]

            for person in

            node["assignees"]["nodes"]

        ]

    )

In [8]:
result = client.execute(

    ISSUES_QUERY,

    {

        "owner": OWNER,

        "name": REPOSITORY

    }

)

In [9]:
nodes = (

    result["repository"]

          ["issues"]

          ["nodes"]

)

issues = [

    build_issue(node)

    for node in nodes

]

print(

    f"Downloaded {len(issues)} issues."

)

Downloaded 100 issues.


In [10]:
if not issues:

    raise RuntimeError(

        "Repository contains no issues."

    )

print(

    "Issue history validated."

)

Issue history validated.


In [11]:
largest_issues = sorted(

    issues,

    key=lambda issue:

        issue.comments,

    reverse=True

)

largest_issue = largest_issues[0]

In [12]:
issues_df = pd.DataFrame(

    [

        {

            "Issue": issue.number,

            "Title": issue.title,

            "Author": issue.author,

            "State": issue.state,

            "Comments": issue.comments,

            "Labels": ", ".join(

                issue.labels

            ),

            "Assignees": ", ".join(

                issue.assignees

            ),

            "Created": issue.created_at,

            "Closed": issue.closed_at

        }

        for issue in largest_issues

    ]

)

issues_df.head(100)

,Issue,Title,Author,State,Comments,Labels,Assignees,Created,Closed
0,36787,[DevTools refactor]: Combine DevTools view-hi...,Biki-das,OPEN,10,"Type: Bug, Status: Unconfirmed, Component: Dev...",,2026-06-14 17:14:32+00:00,NaT
1,36934,experimental_taintUniqueValue throws RangeErro...,HoneyTyagii,OPEN,8,,,2026-07-05 17:41:31+00:00,NaT
2,36749,[DevTools Bug] Cannot read properties of null ...,furyweapons,CLOSED,7,"Type: Bug, Status: Unconfirmed, Component: Dev...",,2026-06-11 08:11:28+00:00,2026-07-10 14:27:38+00:00
3,36569,Bug: React.lazy waits ~300ms before rendering ...,andyticker,OPEN,7,Status: Unconfirmed,,2026-05-29 04:52:54+00:00,NaT
4,36461,Protect react from AI slop PRs,kimjune01,CLOSED,7,,,2026-05-13 07:54:39+00:00,2026-06-25 02:32:33+00:00
...,...,...,...,...,...,...,...,...,...
95,36469,[Compiler Bug]: exp://8dca8a7a-7e6f-42c3-8cdb-...,hwbabasmja-stack,CLOSED,0,"Type: Bug, Status: Unconfirmed",,2026-05-13 23:34:14+00:00,2026-07-09 04:28:32+00:00
96,36459,[DevTools Bug]:,cesarkdksu-prog,CLOSED,0,"Type: Bug, Status: Unconfirmed, Component: Dev...",,2026-05-12 23:47:10+00:00,2026-07-09 04:29:44+00:00
97,36432,[Compiler Bug]: eslint-plugin-react-hooks 7.1....,michaeljaltamirano,OPEN,0,"Type: Bug, Status: Unconfirmed",,2026-05-07 18:53:49+00:00,NaT
98,36430,Bug: [19.2] DEV-build logComponentRender throw...,Rasvom,OPEN,0,Status: Unconfirmed,,2026-05-07 08:55:51+00:00,NaT


In [13]:
state_counts = Counter(

    issue.state

    for issue in issues

)

author_counts = Counter(

    issue.author

    for issue in issues
)

label_counts = Counter(

    label

    for issue in issues

    for label in issue.labels

)

In [14]:
print("=" * 60)

print("Issue Summary")

print("=" * 60)

print(f"Repository      : {OWNER}/{REPOSITORY}")

print(f"Issues          : {len(issues)}")

print(f"Authors         : {len(author_counts)}")

print(f"Open            : {state_counts['OPEN']}")

print(f"Closed          : {state_counts['CLOSED']}")

print(f"Most Comments   : {largest_issue.comments}")

print(

    f"Average Comments: "

    f"{mean(issue.comments for issue in issues):.2f}"

)

print(

    f"Unique Labels   : "

    f"{len(label_counts)}"

)

Issue Summary
Repository      : facebook/react
Issues          : 100
Authors         : 85
Open            : 48
Closed          : 52
Most Comments   : 10
Average Comments: 1.62
Unique Labels   : 4
